In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [3]:
from methods import ShallowOOD
from pipelines import mi_pipelines, ood_pipelines
from processing import *

In [4]:
experiment_mi = "cs_mi_7ch"
experiment_ood = "cs_ood_7ch"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]

datasets = [ "Zhou2016", "BNCI2014_001", "Wang2025", "Yang2025_3C", "Zhou2020" ]


In [5]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None
        
        for method, pipeline in mi_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment_mi, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        data = None
        shallow = None
        for method, pipeline in ood_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment_ood, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, include_feet=True)
                train_data, val_data, test_data = data
            if isinstance(pipeline, ShallowOOD):
                if shallow is None:
                    shallow = train(train_data, mi_pipelines["ShallowConvNet"])

                model = clone(pipeline).fit(*train_data, classifier=shallow)
            else:
                model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display(summarize_results(dataset, experiment_mi, metric="MI_BA").round(3))
    display(summarize_results(dataset, experiment_mi, metric="MI_AUROC").round(3))
    display(summarize_results(dataset, experiment_ood, metric="OOD_AUROC").round(3))

Zhou2016


,metric,n
method,,
ACM+TS+SVM,0.741,4
EEGNet,0.873,4
FBCSP+ElasticNet,0.821,4
Log-bandpower+shLDA,0.734,4
Microstates+GB,0.767,4
ShallowConvNet,0.820,4
VAR+shLDA,0.663,4


,metric,n
method,,
ACM+TS+SVM,0.885,4
EEGNet,0.955,4
FBCSP+ElasticNet,0.925,4
Log-bandpower+shLDA,0.856,4
Microstates+GB,0.833,4
ShallowConvNet,0.903,4
VAR+shLDA,0.736,4


,metric,n
method,,
ACM+TS+OCSVM,0.710,4
EEGNet+TempDens,0.832,4
FBCSP+IsolationForest,0.792,4
Log-bandpower+Mahalanobis,0.684,4
Microstates+LOF,0.726,4
ShallowConvNet+kNN,0.728,4
VAR+Mahalanobis,0.695,4


BNCI2014_001


,metric,n
method,,
ACM+TS+SVM,0.745,9
EEGNet,0.755,9
FBCSP+ElasticNet,0.738,9
Log-bandpower+shLDA,0.677,9
Microstates+GB,0.664,9
ShallowConvNet,0.739,9
VAR+shLDA,0.658,9


... (10 outputs left out)

Zhou2020


,metric,n
method,,
ACM+TS+SVM,0.530,20
EEGNet,0.576,20
FBCSP+ElasticNet,0.562,20
Log-bandpower+shLDA,0.554,20
Microstates+GB,0.550,20
ShallowConvNet,0.579,20
VAR+shLDA,0.526,20


,metric,n
method,,
ACM+TS+SVM,0.602,20
EEGNet,0.605,20
FBCSP+ElasticNet,0.602,20
Log-bandpower+shLDA,0.566,20
Microstates+GB,0.579,20
ShallowConvNet,0.612,20
VAR+shLDA,0.557,20


,metric,n
method,,
ACM+TS+OCSVM,0.517,20
EEGNet+TempDens,0.615,20
FBCSP+IsolationForest,0.600,20
Log-bandpower+Mahalanobis,0.539,20
Microstates+LOF,0.530,20
ShallowConvNet+kNN,0.578,20
VAR+Mahalanobis,0.546,20


In [6]:
from pathlib import Path
import pandas as pd

MI_METHODS = mi_pipelines.keys()
OOD_METHODS = ood_pipelines.keys()

parts = []
for task, experiment, metric in [
    ("MI BA", experiment_mi, "MI_BA"),
    ("MI AUROC", experiment_mi, "MI_AUROC"),
    ("OOD", experiment_ood, "OOD_AUROC") ]:
    d = pd.concat([pd.read_csv(p) for p in (Path("results") / experiment).glob("*.csv")], ignore_index=True)
    methods = OOD_METHODS if task == "OOD" else MI_METHODS
    d = d[d["method"].isin(methods)]
    parts.append(d.assign(task=task, score=d[metric]))
r = pd.concat(parts, ignore_index=True)
means = r.groupby(["task", "method", "dataset"])["score"].mean().unstack("dataset")
table = means.join(r.groupby(["task", "method"])["score"].agg(mean_subjects="mean", sd="std", n="count"))
table["mean_datasets"] = means.mean(axis=1)
display(table.sort_values(["task", "mean_subjects"], ascending=[True, False]).round(3))


BNCI2014_001    TTK  Wang2025  \
task     method                                                     
MI AUROC EEGNet                            0.814  0.686     0.800   
         FBCSP+ElasticNet                  0.858  0.708     0.753   
         ACM+TS+SVM                        0.826  0.731     0.763   
         ShallowConvNet                    0.808  0.639     0.756   
         Log-bandpower+shLDA               0.725  0.652     0.679   
         Microstates+GB                    0.725  0.611     0.648   
         VAR+shLDA                         0.723  0.623     0.654   
MI BA    EEGNet                            0.755  0.633     0.722   
         ShallowConvNet                    0.739  0.589     0.677   
         FBCSP+ElasticNet                  0.738  0.626     0.661   
         ACM+TS+SVM                        0.745  0.624     0.662   
         Log-bandpower+shLDA               0.677  0.601     0.618   
         Microstates+GB                    0.664  0.593     0.603   
         VAR+shLDA                         0.658  0.578     0.610   
OOD      EEGNet+TempDens                   0.748  0.757     0.785   
         FBCSP+IsolationForest             0.709  0.780     0.760   
         ShallowConvNet+kNN                0.683  0.779     0.724   
         Log-bandpower+Mahalanobis         0.590  0.627     0.628   
         ACM+TS+OCSVM                      0.638  0.571     0.633   
         VAR+Mahalanobis                   0.507  0.626     0.604   
         Microstates+LOF                   0.551  0.565     0.604   

                                    Yang2025_3C  Zhou2016  Zhou2020  \
task     method                                                       
MI AUROC EEGNet                           0.640     0.955     0.605   
         FBCSP+ElasticNet                 0.596     0.925     0.602   
         ACM+TS+SVM                       0.557     0.885     0.602   
         ShallowConvNet                   0.596     0.903     0.612   
         Log-bandpower+shLDA              0.577     0.856     0.566   
         Microstates+GB                   0.544     0.833     0.579   
         VAR+shLDA                        0.577     0.736     0.557   
MI BA    EEGNet                           0.599     0.873     0.576   
         ShallowConvNet                   0.553     0.820     0.579   
         FBCSP+ElasticNet                 0.553     0.821     0.562   
         ACM+TS+SVM                       0.515     0.741     0.530   
         Log-bandpower+shLDA              0.552     0.734     0.554   
         Microstates+GB                   0.535     0.767     0.550   
         VAR+shLDA                        0.559     0.663     0.526   
OOD      EEGNet+TempDens                  0.636     0.832     0.615   
         FBCSP+IsolationForest            0.621     0.792     0.600   
         ShallowConvNet+kNN               0.511     0.728     0.578   
         Log-bandpower+Mahalanobis        0.545     0.684     0.539   
         ACM+TS+OCSVM                     0.529     0.710     0.517   
         VAR+Mahalanobis                  0.520     0.695     0.546   
         Microstates+LOF                  0.507     0.726     0.530   

                                    mean_subjects     sd   n  mean_datasets  
task     method                                                              
MI AUROC EEGNet                             0.714  0.175  65          0.750  
         FBCSP+ElasticNet                   0.701  0.169  65          0.740  
         ACM+TS+SVM                         0.692  0.166  65          0.727  
         ShallowConvNet                     0.690  0.154  65          0.719  
         Log-bandpower+shLDA                0.642  0.131  65          0.676  
         Microstates+GB                     0.628  0.139  65          0.657  
         VAR+shLDA                          0.623  0.118  65          0.645  
MI BA    EEGNet                             0.662  0.145  65          0.693  
         ShallowConvNet           